# Calculating and inspecting interactions

Use named analyses to retain scientific data while changing their visual presentation. This tutorial needs the experimental MolSysMT Interactions/H5MSM backend. It uses a bundled system without addon registration.


In [1]:
import tempfile
from pathlib import Path

import molsysviewer as msv

view = msv.new_view(msv.demo["pentalanine"].molsys, structure_indices=[0, 8, 3])
molsys = view.molsys


## Calculating a named analysis

The loaded local frames 0, 1 and 2 correspond to source frames 0, 8 and 3. Calculate Buch hydrogen bonds on local frames 0 and 2, with an explicit hydrogen–acceptor threshold of 2.3 Å. Frame 1 remains unevaluated. The scientific result is stored in `molsys.interactions`.


In [2]:
view.interactions.hbonds.get_buch_hbonds(
    name="buch", structure_indices=[0, 2], distance_threshold="2.3 angstroms", pbc=False
)
view.interactions.analyses()


[{'name': 'buch',
  'interaction_types': [],
  'method': 'molsysmt.interactions.hbonds.get_buch_hbonds',
  'parameters': {'distance_threshold_nm': 0.22999999999999995,
   'distance_definition': 'hydrogen_acceptor',
   'pbc': False,
   'exclude_self_donor_acceptor': True,
   'role_identification': {'donor_inclusion_rules': ["(atom_type=='O') bonded to (atom_type=='H')",
     "(atom_type=='N') bonded to (atom_type=='H')"],
    'donor_exclusion_rules': ["(atom_name=='NE2') not bonded to (atom_type=='H')",
     "(atom_name=='ND1') not bonded to (atom_type=='H')"],
    'acceptor_inclusion_rules': ["atom_type=='O'",
     "atom_type=='N'",
     "atom_type=='S'"],
    'acceptor_exclusion_rules': ["atom_name=='NE2' and group_name=='GLN'",
     "(atom_name=='NE2' and group_name=='HIS') bonded to (atom_type=='H')",
     "(atom_name=='ND1' and group_name=='HIS') bonded to (atom_type=='H')"],
    'hydrogens': 'covalently_bonded_to_selected_donors'},
   'image_policy': 'donor_hydrogen_then_hydrogen_

## Creating and filtering a visual set

Adding a set does not calculate interactions. Changing its selection filters the saved observations. Incident mode includes interactions in which selected atoms participate; internal mode requires every participant atom to belong to the selection. The radius below is 0.025 nm (0.25 Å).


In [3]:
hbonds = view.interactions.add("buch", tag="hbonds", interaction_types=["hbond"])
hbonds.set_color("#34d399")
hbonds.set_radius("0.025 nm")
hbonds.set_filter(selection=[0, 1, 2, 3, 4, 5], mode="incident")
view.interactions.inspect("hbonds", structure_index=0, offset=0, limit=10)


{'tag': 'hbonds',
 'analysis_name': 'buch',
 'analysis_revision': 'sha256:02099742293ff6eabc75e021f42019d5354f85b33fcea21943dd9d8a2eedc22f',
 'query_revision': 'sha256:b02b2fc2aba8732a154ca502dd08cf85716304000fd54c91152d363d167f8ce1',
 'frame': 0,
 'status': 'evaluated',
 'method': 'molsysmt.interactions.hbonds.get_buch_hbonds',
 'parameters': {'distance_threshold_nm': 0.22999999999999995,
  'distance_definition': 'hydrogen_acceptor',
  'pbc': False,
  'exclude_self_donor_acceptor': True,
  'role_identification': {'donor_inclusion_rules': ["(atom_type=='O') bonded to (atom_type=='H')",
    "(atom_type=='N') bonded to (atom_type=='H')"],
   'donor_exclusion_rules': ["(atom_name=='NE2') not bonded to (atom_type=='H')",
    "(atom_name=='ND1') not bonded to (atom_type=='H')"],
   'acceptor_inclusion_rules': ["atom_type=='O'",
    "atom_type=='N'",
    "atom_type=='S'"],
   'acceptor_exclusion_rules': ["atom_name=='NE2' and group_name=='GLN'",
    "(atom_name=='NE2' and group_name=='HIS') 

In [4]:
unevaluated = view.interactions.inspect("hbonds", structure_index=1)
assert unevaluated["status"] == "unevaluated"
unevaluated["status"]


'unevaluated'

## Querying saved scientific data

Queries use local atom and structure indices. This nonconsecutive list does not copy the full trajectory. `to_dict()` currently materializes all selected observations, so use bounded selections when requesting that representation.


In [5]:
query = view.interactions.query("buch", structure_indices=[2, 0], selection=[0, 1, 2])
query.n_interactions


0

## Loading a named analysis from H5MSM

An independent H5MSM can contain only named interactions. Declare atom/structure correspondence explicitly when loading it; matching dimensions do not authenticate the origin. Here the file was written from this same system, so its order is known. A session preserves both molecular data and visual references. All files below are temporary.


In [6]:
import molsysmt as msm

with tempfile.TemporaryDirectory(prefix="msv-interactions-tutorial-") as directory:
    filename = Path(directory) / "analyses.h5msm"
    msm.h5msm.write_layers(str(filename), interactions={"buch": view.interactions.get_analysis("buch")})
    view.interactions.load(str(filename), analysis_name="buch", name="imported-buch", assume_aligned=True)
    imported = view.interactions.add("imported-buch", tag="imported")
    imported.hide()
    view.save_session(str(Path(directory) / "interactions.msv"))

assert "imported-buch" in molsys.interactions


## Working in Studio

Display the view and open Interactions in the floating card. Calculate, Stored analysis and H5MSM file use the same Python methods. The saved cards offer filters, style, visibility, focus, inspection and deletion. Deleting a set retains the named analysis. Disulfide calculations produce geometric candidates rather than topology bonds. Large materialization requests report an explicit limit; file-backed queries and additional scientific families remain future work.


In [7]:
view
